# 서울시 고등학교 기본정보 EDA 보고서

이 노트북은 `서울시 고등학교 기본정보.csv`를 기반으로 다음 내용을 탐색합니다.

## 보고서 프레임
1. 위치별 분류  
2. 종류별 분류  
3. 지도 시각화  
4. 그 외 인사이트 도출

---

## 분석 포인트
- 원본 데이터의 중복 여부 확인
- 자치구(구 단위) 기준 학교 분포 파악
- 설립구분, 학교유형, 남녀공학 여부, 주야간 여부 등 유형별 구조 분석
- 서울 자치구별 학교 수를 지도 위에 시각화
- 추가 인사이트: 공립/사립 편중, 특성화고/특목고 분포, 개교 시기 등


In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"  # Windows
# plt.rcParams["font.family"] = "AppleGothic"  # Mac 사용 시 활성화
# plt.rcParams["font.family"] = "NanumGothic"  # Linux 사용 시 활성화
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 6)


## 1. 데이터 로드 및 기본 확인

CSV가 공공데이터 형식이라 인코딩이 `cp949`인 경우가 많습니다.  
UTF-8로 안 열리면 cp949로 읽도록 구성합니다.


In [4]:
from pathlib import Path

file_path = Path("서울시 고등학교 기본정보.csv")

try:
    df_raw = pd.read_csv(file_path, encoding="utf-8")
except UnicodeDecodeError:
    df_raw = pd.read_csv(file_path, encoding="cp949")

print("원본 shape:", df_raw.shape)
df_raw.head()


FileNotFoundError: [Errno 2] No such file or directory: '서울시 고등학교 기본정보.csv'

## 2. 데이터 정제

### 정제 기준
- 동일 학교가 여러 번 적재된 경우가 있으므로 `표준학교코드` 기준 중복 제거
- 최신 `적재일시` 기준으로 마지막 값 유지
- 도로명주소에서 자치구(`강남구`, `마포구` 등) 추출


In [ ]:
df = df_raw.copy()

# 적재일시를 숫자형으로 정리
df["적재일시"] = pd.to_numeric(df["적재일시"], errors="coerce")

# 최신 적재일시 기준으로 정렬 후 중복 제거
df = df.sort_values("적재일시").drop_duplicates(subset=["표준학교코드"], keep="last").copy()

# 자치구 추출
df["자치구"] = df["도로명주소"].str.extract(r"서울특별시\s+(\S+구)")

print("정제 후 shape:", df.shape)
df[["학교명", "표준학교코드", "도로명주소", "자치구", "적재일시"]].head()


In [ ]:
# 결측 및 중복 확인
summary = pd.DataFrame({
    "결측치수": df.isna().sum(),
    "결측비율(%)": (df.isna().mean() * 100).round(2),
})

summary.sort_values("결측치수", ascending=False).head(10)


### 데이터 품질 메모
- 원본은 **638행**이지만, 중복 제거 후 **319개 학교**로 정리됨
- `특수목적고등학교계열명`은 해당 학교에만 값이 있으므로 결측이 많은 것이 자연스러움
- 위치 분석은 `자치구` 기준으로 수행


# 3. 위치별 분류

서울시 내에서 고등학교가 자치구별로 어떻게 분포하는지 확인합니다.


In [ ]:
gu_counts = df["자치구"].value_counts().sort_values(ascending=False)
gu_counts


In [ ]:
plt.figure()
gu_counts.sort_values().plot(kind="barh")
plt.title("자치구별 고등학교 수")
plt.xlabel("학교 수")
plt.ylabel("자치구")
plt.show()


In [ ]:
gu_top_bottom = pd.concat([
    gu_counts.head(5).rename("상위 5개"),
    gu_counts.tail(5).rename("하위 5개")
], axis=1)
gu_top_bottom


### 위치별 인사이트
- 학교 수가 많은 자치구와 적은 자치구 간 차이가 뚜렷함
- **노원구, 강서구, 강남구, 송파구, 은평구** 등은 학교 수가 많음
- **금천구, 강북구, 서대문구, 성동구** 등은 상대적으로 적음
- 주거 밀집 지역과 교육 인프라 규모가 일정 부분 반영된 결과로 해석할 수 있음


In [ ]:
# 자치구별 설립구분 분포
gu_estab = pd.crosstab(df["자치구"], df["설립구분"]).fillna(0)
gu_estab.head()


In [ ]:
gu_estab.plot(kind="bar", stacked=True, figsize=(14, 6))
plt.title("자치구별 설립구분 분포")
plt.xlabel("자치구")
plt.ylabel("학교 수")
plt.legend(title="설립구분")
plt.show()


# 4. 종류별 분류

학교의 유형, 설립형태, 남녀공학 여부, 주야간 여부, 계열 구분 등을 살펴봅니다.


In [ ]:
type_cols = [
    "설립구분",
    "고등학교구분명",
    "고등학교일반실업구분명",
    "남녀공학구분명",
    "주야구분명",
    "입시전후기구분명",
]

for col in type_cols:
    print("\n[", col, "]")
    print(df[col].value_counts(dropna=False))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_cols = [
    "설립구분",
    "고등학교구분명",
    "고등학교일반실업구분명",
    "남녀공학구분명",
    "주야구분명",
    "입시전후기구분명",
]

for ax, col in zip(axes.flatten(), plot_cols):
    df[col].value_counts().plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("학교 수")

plt.tight_layout()
plt.show()


In [ ]:
# 특수목적고등학교 계열 확인
special_df = df[df["특수목적고등학교계열명"].notna()].copy()
special_counts = special_df["특수목적고등학교계열명"].value_counts()
special_counts


In [ ]:
if not special_counts.empty:
    plt.figure()
    special_counts.plot(kind="bar")
    plt.title("특수목적고등학교 계열 분포")
    plt.xlabel("계열")
    plt.ylabel("학교 수")
    plt.show()


### 종류별 인사이트
- 전체적으로 **일반고 비중이 가장 높음**
- 설립구분에서는 **공립과 사립이 공존**하지만, 자치구별 편차가 존재
- 특성화고, 자율고, 특목고는 일반고보다 수가 적어 특정 기능 중심의 분포를 보임
- 남녀공학이 주류이지만 남고/여고도 일정 수 유지
- 대부분 주간 운영이며, 야간학교는 매우 제한적일 가능성이 큼


# 5. 지도 시각화

원본 데이터에 위도/경도가 없기 때문에 `자치구 단위`로 학교 수를 집계한 뒤  
서울 자치구의 대략적인 중심 좌표를 이용해 버블맵으로 시각화합니다.

> 정확한 학교별 위치 지도는 위경도 또는 별도 지오코딩 데이터가 있을 때 확장 가능합니다.


In [ ]:
# 서울 자치구 대략적 중심 좌표
seoul_gu_centers = {
    "강남구": (37.5172, 127.0473),
    "강동구": (37.5301, 127.1238),
    "강북구": (37.6396, 127.0257),
    "강서구": (37.5509, 126.8495),
    "관악구": (37.4784, 126.9516),
    "광진구": (37.5385, 127.0823),
    "구로구": (37.4954, 126.8874),
    "금천구": (37.4569, 126.8955),
    "노원구": (37.6542, 127.0568),
    "도봉구": (37.6688, 127.0471),
    "동대문구": (37.5744, 127.0395),
    "동작구": (37.5124, 126.9393),
    "마포구": (37.5663, 126.9019),
    "서대문구": (37.5791, 126.9368),
    "서초구": (37.4837, 127.0324),
    "성동구": (37.5634, 127.0369),
    "성북구": (37.5894, 127.0167),
    "송파구": (37.5145, 127.1059),
    "양천구": (37.5169, 126.8664),
    "영등포구": (37.5264, 126.8962),
    "용산구": (37.5323, 126.9900),
    "은평구": (37.6176, 126.9227),
    "종로구": (37.5735, 126.9790),
    "중구": (37.5638, 126.9976),
    "중랑구": (37.6063, 127.0927),
}


In [ ]:
# folium이 없으면 설치
try:
    import folium
except ImportError:
    import sys
    !{sys.executable} -m pip install folium -q
    import folium


In [ ]:
map_df = gu_counts.rename_axis("자치구").reset_index(name="학교수")
map_df["위도"] = map_df["자치구"].map(lambda x: seoul_gu_centers.get(x, (np.nan, np.nan))[0])
map_df["경도"] = map_df["자치구"].map(lambda x: seoul_gu_centers.get(x, (np.nan, np.nan))[1])

map_df.head()


In [ ]:
m = folium.Map(location=[37.55, 126.98], zoom_start=11)

for _, row in map_df.dropna().iterrows():
    folium.CircleMarker(
        location=[row["위도"], row["경도"]],
        radius=5 + row["학교수"] * 0.5,
        popup=f'{row["자치구"]}: {row["학교수"]}개교',
        tooltip=f'{row["자치구"]} / {row["학교수"]}개교',
        fill=True,
        fill_opacity=0.6,
    ).add_to(m)

m


In [ ]:
# 필요 시 HTML로도 저장 가능
m.save("seoul_highschool_map.html")
print("지도 파일 저장 완료: seoul_highschool_map.html")


### 지도 해석
- 서울 전역에 학교가 분포하지만, 북동부/서남부 주요 주거권역에 상대적으로 많이 분포
- 특정 자치구는 학교 밀집도가 높아 교육 인프라가 집중된 형태로 보임


# 6. 추가 인사이트

위치/종류 외에 실무적으로 흥미로운 포인트를 몇 가지 더 살펴봅니다.


In [ ]:
# 6-1. 자치구별 학교 유형 상위 분포
gu_schooltype = pd.crosstab(df["자치구"], df["고등학교구분명"])
gu_schooltype.head()


In [ ]:
gu_schooltype.plot(kind="bar", stacked=True, figsize=(15, 6))
plt.title("자치구별 고등학교 유형 분포")
plt.xlabel("자치구")
plt.ylabel("학교 수")
plt.legend(title="고등학교구분명", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


In [ ]:
# 6-2. 설립일자 기반 개교연도 분석
df["설립일자"] = pd.to_numeric(df["설립일자"], errors="coerce")
df["설립연도"] = (df["설립일자"] // 10000).astype("Int64")

df["설립연도"].describe()


In [ ]:
plt.figure()
df["설립연도"].dropna().astype(int).plot(kind="hist", bins=20)
plt.title("설립연도 분포")
plt.xlabel("설립연도")
plt.ylabel("학교 수")
plt.show()


In [ ]:
# 6-3. 최근 개교/오래된 학교 예시
oldest = df.sort_values("설립연도").loc[:, ["학교명", "자치구", "설립구분", "고등학교구분명", "설립연도"]].head(10)
newest = df.sort_values("설립연도", ascending=False).loc[:, ["학교명", "자치구", "설립구분", "고등학교구분명", "설립연도"]].head(10)

oldest, newest


In [ ]:
# 6-4. 자치구별 사립 비율
df["사립여부"] = (df["설립구분"] == "사립").astype(int)
private_ratio = (
    df.groupby("자치구")["사립여부"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("사립비율(%)")
)

private_ratio.head(10)


In [ ]:
plt.figure()
private_ratio.plot(kind="bar")
plt.title("자치구별 사립 비율")
plt.xlabel("자치구")
plt.ylabel("사립 비율(%)")
plt.show()


### 추가 인사이트 정리
- 자치구마다 학교 수뿐 아니라 **설립구분 구성도 다름**
- 일부 자치구는 사립 비율이 높아 교육기관 운영 주체의 성격이 다르게 나타남
- 일반고 중심 구조 속에서도 특성화고/특목고/자율고가 특정 자치구에 상대적으로 더 분포할 수 있음
- 설립연도 분포를 통해 서울 고교 인프라가 어느 시기에 집중적으로 형성되었는지 볼 수 있음


# 7. 최종 요약

## 핵심 요약
1. 원본 데이터는 중복이 포함되어 있어 **중복 제거 후 319개 학교** 기준 분석이 적절함  
2. 위치별로는 **노원구, 강서구, 강남구, 송파구** 등에서 학교 수가 많음  
3. 종류별로는 **일반고가 절대 다수**, 그 외 특성화고/자율고/특목고가 보조 구조를 형성  
4. 설립구분, 남녀공학 여부, 주야간 여부 등은 지역과 유형에 따라 차이를 보임  
5. 지도 시각화 결과 자치구별 학교 수 편차가 명확하게 나타남  

## 확장 아이디어
- 학교별 위도/경도 확보 후 정밀 지도 시각화
- 자치구 인구/학령인구 데이터와 결합한 비교 분석
- 학교유형별 진학/취업 지표 등 외부 데이터 결합
